# DeepSeek-Math-7B-Instruct (SFT): CommonsenseQA pass@8 — 200 examples rerun

In [ ]:
!pip install -q --upgrade transformers accelerate datasets sentencepiece tqdm protobuf safetensors
!pip install -q --upgrade bitsandbytes

In [ ]:
import gc
import re
import json
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ── Config ──────────────────────────────────────────────────────────────────
MODEL_NAME   = "deepseek-ai/deepseek-math-7b-instruct"
DISPLAY_NAME = "DeepSeek-Math-7B-Instruct"
STAGE        = "SFT / Instruct"

CSQA_LIMIT       = 200
NUM_SAMPLES      = 8
TEMPERATURE      = 0.7
TOP_P            = 0.95
MAX_NEW_TOKENS   = 32

OUTPUT_DIR = Path("deepseek_sft_csqa_pass8_rerun")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 1234
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"Model : {MODEL_NAME}")
print(f"CSQA  : first {CSQA_LIMIT} validation examples")
print(f"pass@k: k={NUM_SAMPLES}, temp={TEMPERATURE}, top_p={TOP_P}")
print(f"Output: {OUTPUT_DIR.resolve()}")

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────────
csqa_full = load_dataset("tau/commonsense_qa", split="validation")
csqa = csqa_full.select(range(min(CSQA_LIMIT, len(csqa_full))))
print(f"CommonsenseQA validation (first {CSQA_LIMIT}):", csqa)
print("Example:", csqa[0])

In [ ]:
# ── Load model (4-bit NF4) ────────────────────────────────────────────────────
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.eval()
print("Loaded:", MODEL_NAME)

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────
def build_prompt(ex: Dict[str, Any]) -> str:
    labels = ex["choices"]["label"]
    texts  = ex["choices"]["text"]
    options = "\n".join(f"{l}. {t}" for l, t in zip(labels, texts))
    return (
        "Answer the following multiple-choice question.\n"
        "Choose the best answer from A, B, C, D, and E.\n"
        "Return only one letter: A, B, C, D, or E.\n\n"
        f"Question: {ex['question']}\n"
        f"{options}\n\n"
        "Answer:"
    )


def extract_answer(text: str) -> Optional[str]:
    if not text:
        return None
    t = text.upper().replace("Ġ", " ").replace("Ċ", "\n").strip()
    for pattern in [
        r"\\BOXED\{\s*([ABCDE])\s*\}",
        r"ANSWER\s*(?:IS)?\s*[:\-]?\s*([ABCDE])\b",
        r"^\s*([ABCDE])\s*(?:[\.):\-]|\b)",
        r"\b([ABCDE])\b",
    ]:
        m = re.search(pattern, t)
        if m:
            return m.group(1)
    return None


def generate_one(prompt: str, temperature: float) -> str:
    wrapped = f"User: {prompt}\nAssistant:"
    inputs  = tokenizer(wrapped, return_tensors="pt", return_token_type_ids=False)
    device  = next(model.parameters()).device
    inputs  = {k: v.to(device) for k, v in inputs.items()}
    do_sample = temperature > 0
    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=do_sample,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = TOP_P
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
# ── Evaluation loop ───────────────────────────────────────────────────────────
predictions = []
pass1_correct = 0
pass8_correct = 0
pass1_parsed  = 0
any_parsed    = 0
total = len(csqa)

print(f"Starting CommonsenseQA pass@{NUM_SAMPLES} | total={total}")

for i, ex in enumerate(csqa, start=1):
    if i == 1 or i % 25 == 0 or i == total:
        print(f"  {i}/{total}", flush=True)

    prompt = build_prompt(ex)
    gold   = ex["answerKey"]

    raw_outputs = [generate_one(prompt, TEMPERATURE) for _ in range(NUM_SAMPLES)]
    preds       = [extract_answer(o) for o in raw_outputs]

    p1_pred = preds[0]
    if p1_pred is not None:
        pass1_parsed += 1
    if any(p is not None for p in preds):
        any_parsed += 1
    if p1_pred == gold:
        pass1_correct += 1
    if any(p == gold for p in preds):
        pass8_correct += 1

    predictions.append({
        "id":                  ex["id"],
        "question":            ex["question"],
        "gold":                gold,
        "sampled_predictions": preds,
        "sampled_raw_outputs": raw_outputs,
        "pass1_sample0_correct": p1_pred == gold,
        "pass8_correct":         any(p == gold for p in preds),
    })

metrics = {
    "model":                        MODEL_NAME,
    "display_name":                 DISPLAY_NAME,
    "stage":                        STAGE,
    "benchmark":                    "CommonsenseQA",
    "split":                        "validation",
    "num_examples":                 total,
    "num_samples":                  NUM_SAMPLES,
    "pass_at_1_sample0_accuracy":   pass1_correct / total,
    "pass_at_8_accuracy":           pass8_correct / total,
    "pass_at_1_sample0_parse_rate": pass1_parsed  / total,
    "any_sample_parse_rate":        any_parsed    / total,
    "sampling_temperature":         TEMPERATURE,
    "sampling_top_p":               TOP_P,
}

suffix = f"{total}_n{NUM_SAMPLES}"
pred_file    = OUTPUT_DIR / f"commonsenseqa_pass8_predictions_{suffix}.json"
metrics_file = OUTPUT_DIR / f"commonsenseqa_pass8_metrics_{suffix}.json"

with open(pred_file, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
with open(metrics_file, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("\n=== Results ===")
print(json.dumps(metrics, indent=2))
print("Saved:", pred_file)
print("Saved:", metrics_file)

In [ ]:
# ── Spot-check first 3 examples ──────────────────────────────────────────────
for item in predictions[:3]:
    print("=" * 70)
    print("Q   :", item["question"])
    print("Gold:", item["gold"])
    print("Pred:", item["sampled_predictions"])
    print("pass@8 correct:", item["pass8_correct"])
    print("Sample 0 raw   :", item["sampled_raw_outputs"][0][:200])